In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

SEED = 42
TEST_SIZE = 0.2
N_SPLITS = 5

In [2]:
df_path = Path("data/silver.parquet")
df = pd.read_parquet(df_path).set_index("customer_id")

print(df.shape)
df.head()

(3682, 14)


,first_purchase,last_purchase,n_purchases,median_gap,total_revenue,avg_order_value,total_units,avg_basket_size,avg_unique_items,last_gap_vs_median,recency,tenure,recency_vs_gap,churn
customer_id,,,,,,,,,,,,,,
12346.0,2009-12-14,2011-01-18,12,8.0,77556.46,6463.038333,74285,6190.416667,2.833333,25.5000,235,635,29.375000,1
12347.0,2010-10-31,2011-08-02,6,54.0,4114.18,685.696667,2418,403.000000,27.333333,1.0000,39,314,0.722222,0
12348.0,2010-09-27,2011-04-05,4,70.0,1388.40,347.100000,2488,622.000000,10.000000,1.0000,158,348,2.257143,0
12349.0,2010-04-29,2010-10-28,2,182.0,2221.14,1110.570000,991,495.500000,50.000000,1.0000,317,499,1.741758,0
12352.0,2010-11-12,2011-03-22,6,16.0,985.31,164.218333,437,72.833333,8.500000,0.3125,172,302,10.750000,0


In [3]:
DROP_COLS = ["first_purchase", "last_purchase", "total_units"]

model_feats = df.drop(columns=DROP_COLS)
print(model_feats.shape)
model_feats.head()

(3682, 11)


,n_purchases,median_gap,total_revenue,avg_order_value,avg_basket_size,avg_unique_items,last_gap_vs_median,recency,tenure,recency_vs_gap,churn
customer_id,,,,,,,,,,,
12346.0,12,8.0,77556.46,6463.038333,6190.416667,2.833333,25.5000,235,635,29.375000,1
12347.0,6,54.0,4114.18,685.696667,403.000000,27.333333,1.0000,39,314,0.722222,0
12348.0,4,70.0,1388.40,347.100000,622.000000,10.000000,1.0000,158,348,2.257143,0
12349.0,2,182.0,2221.14,1110.570000,495.500000,50.000000,1.0000,317,499,1.741758,0
12352.0,6,16.0,985.31,164.218333,72.833333,8.500000,0.3125,172,302,10.750000,0


# Exploration

In [4]:
corr = model_feats.drop(columns=["churn"]).corr()
print("Highly correlated feature pairs (|r| > 0.85):")
pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .loc[lambda s: s.abs() > 0.85]
    .sort_values(ascending=False)
)
print(pairs)
# Dropped total_units since it's highly correlated with total_spent and price, and price is more directly related to the target.

Highly correlated feature pairs (|r| > 0.85):
Series([], dtype: float64)


In [5]:
# Look for simple relationships between features and target by comparing churn rates above/below median for each feature.

for col in model_feats.drop(columns=["churn"]).columns:
    med = model_feats[col].median()
    low = model_feats.loc[model_feats[col] <= med, "churn"].mean()
    high = model_feats.loc[model_feats[col] > med, "churn"].mean()
    print(f"{col:18s}  churn below median={low:.2f}  above={high:.2f}")

n_purchases         churn below median=0.62  above=0.30
median_gap          churn below median=0.39  above=0.54
total_revenue       churn below median=0.62  above=0.31
avg_order_value     churn below median=0.52  above=0.41
avg_basket_size     churn below median=0.53  above=0.40
avg_unique_items    churn below median=0.51  above=0.42
last_gap_vs_median  churn below median=0.51  above=0.40
recency             churn below median=0.30  above=0.64
tenure              churn below median=0.49  above=0.44
recency_vs_gap      churn below median=0.37  above=0.57


In [6]:
model_feats.drop(columns=["churn"]).skew().sort_values(ascending=False)

avg_basket_size       43.614288
last_gap_vs_median    26.804389
total_revenue         20.574693
avg_order_value       14.541835
n_purchases            9.424581
recency_vs_gap         6.883249
median_gap             2.109855
avg_unique_items       1.954888
recency                0.924583
tenure                -0.928592
dtype: Float64

# Model Prep

In [ ]:
X = model_feats.drop(columns=["churn"])
y = model_feats["churn"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
)

print(f"Train {X_tr.shape}, test {X_te.shape}, churn rate {y.mean():.3f}")

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

Train (2945, 10), test (737, 10), churn rate 0.466


# Logistic Regression - Baseline Model

In [ ]:
LOG_COLS = [
    "total_revenue",
    "avg_order_value",
    "avg_basket_size",
    "n_purchases",
    "last_gap_vs_median",
]
NUM_COLS = [c for c in X.columns if c not in LOG_COLS]

In [ ]:
prep = ColumnTransformer(
    transformers=[
        (
            "log",
            Pipeline(
                [
                    ("log", FunctionTransformer(np.log1p)),
                ]
            ),
            LOG_COLS,
        ),
    ],
    remainder="passthrough",
)

logit = Pipeline(
    [
        ("prep", prep),
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=SEED)),
    ]
)


In [13]:
logit_cv = cross_val_score(logit, X_tr, y_tr, cv=cv, scoring="roc_auc")
print(f"Logistic CV AUC: {logit_cv.mean():.3f} +/- {logit_cv.std():.3f}")
logit.fit(X_tr, y_tr)

Logistic CV AUC: 0.770 +/- 0.020


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('scale', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['n_purchases','median_gap','total_revenue',...,'recency','tenure', 'recency_vs_gap']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('log', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all rem

In [15]:
rem_cols = [c for c in X.columns if c not in LOG_COLS]
coef = pd.Series(logit.named_steps["clf"].coef_[0], index=LOG_COLS + rem_cols)
print("Logistic coefficients (scaled features):")
print(coef)

Logistic coefficients (scaled features):
total_revenue         0.388129
avg_order_value      -0.332124
avg_basket_size      -0.015716
n_purchases          -0.902459
last_gap_vs_median    0.158481
median_gap            0.204706
avg_unique_items     -0.180090
recency               0.596076
tenure                0.122827
recency_vs_gap        0.043269
dtype: float64


# Random Forest

In [ ]:
rf = Pipeline(
    [
        ("imp", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)),
    ]
)

In [17]:
rf_cv = cross_val_score(rf, X_tr, y_tr, cv=cv, scoring="roc_auc")
print(f"Random forest CV AUC: {rf_cv.mean():.3f} +/- {rf_cv.std():.3f}")
rf.fit(X_tr, y_tr)

Random forest CV AUC: 0.766 +/- 0.013


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imp', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['n_purchases','median_gap','total_revenue',...,'recency','tenure', 'recency_vs_gap']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or obj

## Feature Tuning - Grid Search

In [20]:
param_dist = {
    "clf__n_estimators": [200, 300, 500],
    "clf__max_depth": [None, 6, 10, 16],
    "clf__min_samples_leaf": [1, 3, 5],
}

In [ ]:
search = RandomizedSearchCV(
    rf,
    param_dist,
    n_iter=10,
    cv=cv,
    scoring="roc_auc",
    random_state=SEED,
    n_jobs=-1,
)

search.fit(X_tr, y_tr)
print(f"Best params: {search.best_params_}")
print(f"Best CV AUC: {search.best_score_:.3f}")
rf = search.best_estimator_

Best params: {'clf__n_estimators': 500, 'clf__min_samples_leaf': 5, 'clf__max_depth': 6}
Best CV AUC: 0.778


In [22]:
importances = pd.Series(
    rf.named_steps["clf"].feature_importances_, index=X.columns
).sort_values(ascending=False)
print("RF feature importances:")
print(importances)

RF feature importances:
recency               0.242221
total_revenue         0.178102
n_purchases           0.159965
recency_vs_gap        0.103172
median_gap            0.097411
tenure                0.057628
avg_order_value       0.044395
avg_basket_size       0.040295
avg_unique_items      0.039342
last_gap_vs_median    0.037469
dtype: float64


# Evaluate on holdout test set

In [ ]:
def precision_at_k(y_true, y_scores, k):
    """Fraction of true churners among the top-k highest-scored customers.
    k as a fraction (<=1) = top k%, or as an int = top-k count."""
    y_true = np.asarray(y_true)
    n = len(y_true)
    k = int(np.ceil(k * n)) if k <= 1 else int(k)
    top = np.argsort(y_scores)[::-1][:k]
    return y_true[top].mean()


def evaluate(name, model, X_te, y_te, ks=(0.1, 0.2, 0.3)):
    proba = model.predict_proba(X_te)[:, 1]
    auc = roc_auc_score(y_te, proba)
    base = y_te.mean()
    print(f"\n{name}")
    print(f"  AUC = {auc:.3f}   (base churn rate = {base:.3f})")
    for k in ks:
        p = precision_at_k(y_te, proba, k)
        print(f"  precision@top-{int(k * 100):>2d}% = {p:.3f}   (lift {p / base:.2f}x)")
    print("  --- classification report @ 0.5 threshold ---")
    print(classification_report(y_te, (proba >= 0.5).astype(int), digits=3))
    return proba

In [24]:
logit_proba = evaluate("Logistic regression", logit, X_te, y_te)
rf_proba = evaluate("Random forest", rf, X_te, y_te)


Logistic regression
  AUC = 0.785   (base churn rate = 0.465)
  precision@top-10% = 0.797   (lift 1.71x)
  precision@top-20% = 0.791   (lift 1.70x)
  precision@top-30% = 0.752   (lift 1.62x)
  --- classification report @ 0.5 threshold ---
              precision    recall  f1-score   support

           0      0.726     0.741     0.734       394
           1      0.696     0.679     0.687       343

    accuracy                          0.712       737
   macro avg      0.711     0.710     0.710       737
weighted avg      0.712     0.712     0.712       737


Random forest
  AUC = 0.795   (base churn rate = 0.465)
  precision@top-10% = 0.865   (lift 1.86x)
  precision@top-20% = 0.784   (lift 1.68x)
  precision@top-30% = 0.766   (lift 1.65x)
  --- classification report @ 0.5 threshold ---
              precision    recall  f1-score   support

           0      0.751     0.711     0.730       394
           1      0.687     0.729     0.707       343

    accuracy                       

In [ ]:
ranked = pd.DataFrame(
    {"churn_proba": rf_proba, "actual_churn": y_te.to_numpy()}, index=X_te.index
).sort_values("churn_proba", ascending=False)
print("Top 10 highest-risk customers:")
print(ranked.head(10))

Top 10 highest-risk customers:
             churn_proba  actual_churn
customer_id                           
13825.0         0.882622             1
16452.0         0.881865             1
13457.0         0.880360             1
12663.0         0.876282             1
16840.0         0.875771             1
14486.0         0.874745             1
13442.0         0.872796             1
15086.0         0.872222             1
15323.0         0.870868             1
15504.0         0.866505             0
